In [1]:
%pip install -U pyobjc-framework-Quartz pyobjc-framework-Vision

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import numpy as np
import glob
import Quartz
import Vision
import CoreFoundation
from Cocoa import NSURL
from Foundation import NSDictionary, NSArray
import matplotlib.pyplot as plt

import requests as req
import re

from glob import glob
from tqdm import tqdm
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import bs4


In [3]:
def make_request_handler_img(results):
    """results: list to store results"""
    if not isinstance(results, list):
        raise ValueError("results must be a list")

    def handler(request, error):
        if error:
            print(f"Error! {error}")
        else:
            observations: "list[Vision.VNRecognizedTextObservation]" = request.results()
            for text_observation in observations:
                recognized_text = text_observation.topCandidates_(1)[0]
                corners = {
                    "tl": text_observation.topLeft(),
                    "tr": text_observation.topRight(),
                    "bl": text_observation.bottomLeft(),
                    "br": text_observation.bottomRight(),
                }
                corners = {k: (v.x, v.y) for k, v in corners.items()}
                results.append(
                    [recognized_text.string(), recognized_text.confidence(), corners]
                )

    return handler


def image_to_text(
    img_path, lang="eng"
) -> "list[tuple[str, float, dict[str,tuple[float,float]]]]":
    input_url = NSURL.fileURLWithPath_(img_path)

    input_image = Quartz.CIImage.imageWithContentsOfURL_(input_url)

    vision_options = NSDictionary.dictionaryWithDictionary_({})

    vision_handler = Vision.VNImageRequestHandler.alloc().initWithCIImage_options_(
        input_image, vision_options
    )
    results = []
    handler = make_request_handler_img(results)
    vision_request = Vision.VNRecognizeTextRequest.alloc().initWithCompletionHandler_(
        handler
    )
    # print(vision_request.recognitionLanguages())
    vision_request.setRecognitionLanguages_(
        NSArray.arrayWithArray_(
            [
                lang,
            ]
        )
    )
    # vision_request.setCustomWords_(NSArray.arrayWithArray_(['für',]))
    # print(type(vision_request.recognitionLanguages()))
    # print(vision_request.recognitionLanguages())
    vision_request.setUsesCPUOnly_(False)  # somehow improves accuracy??
    error = vision_handler.performRequests_error_([vision_request], None)

    return results

In [9]:
from pathlib import Path
img = Path("./Poster_Program_FNIPday2026_A3-1-scaled.png")
texts = image_to_text(str(img.absolute()))

In [10]:
texts

[['FOIO',
  0.30000001192092896,
  {'tl': (0.11770467586062357, 0.9754575523258956),
   'tr': (0.35799035853884625, 0.9780252101542266),
   'bl': (0.11875382887067402, 0.9202306034685882),
   'br': (0.3590395115488967, 0.9227982612969191)}],
 ['FOCUS ON OPTICAL',
  1.0,
  {'tl': (0.37561781213270345, 0.9679842363026906),
   'tr': (0.600660123483208, 0.9718201848593522),
   'bl': (0.3761360705453825, 0.9508815484967996),
   'br': (0.601178381895887, 0.9547174970534612)}],
 ['neuro-ImAgIng',
  0.5,
  {'tl': (0.3759689971252204, 0.9534883722402199),
   'tr': (0.5620155020995857, 0.9534883722402199),
   'bl': (0.3759689971252204, 0.9374999999739273),
   'br': (0.5620155020995857, 0.9374999999739273)}],
 ['ANd PHOTONICS',
  1.0,
  {'tl': (0.37596899623341723, 0.9390624998647512),
   'tr': (0.5658914720482365, 0.9390624998647512),
   'bl': (0.37596899623341723, 0.9244186049837412),
   'br': (0.5658914720482365, 0.9244186049837412)}],
 ['SCHEDULE',
  1.0,
  {'tl': (0.021245135560842256, 0.902